# Log & Power Transforms (Box-Cox)

_Feature Engineering_

**Squash a long-tailed positive feature so it looks more like a bell curve.**

A heavy-tailed positive feature is lopsided: a tall pile of small values and a long thin tail of giants.

       The logarithm is a ruler that gets coarser as numbers grow. Going from 1 to 10 and from 1000 to 10000 are the same step in log-land (both multiply by 10).

---

This notebook builds the log / power transform demo up one step at a time. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. Each row is one example; the columns are its features, plus a class label.

In [ ]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer(as_frame=True)
print("rows x columns:", data.frame.shape)
print("feature columns:", list(data.data.columns))
print("target classes :", list(data.target_names))
data.frame.head()

## See it for yourself: the problem, then the fix

Run this cell. It plots the **raw** data and reproduces the problem it causes, then plots the **engineered** data and shows the fix — with before/after numbers.

### Step 1 — Build a heavy-tailed feature with a log-linear target

We construct a realistic right-skewed feature: `x` is log-normal (`exp` of a normal), the shape you get for income, counts, and sizes — most values small, a few enormous. The target is genuinely log-linear, `y = 3 + 2*log(x) + noise`, so the true signal lives in `log(x)`, not `x`. We prepare both the raw feature and its `log1p(x) = log(1 + x)` version as 2-D matrices for scikit-learn.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

rng = np.random.default_rng(0)
n = 400

# x is positive and heavy-tailed: most values small, a few enormous.
x = np.exp(rng.normal(loc=0.0, scale=1.3, size=n))      # log-normal -> right-skewed
# TRUE signal is log-linear: y = a + b*log(x) + noise.
y = 3.0 + 2.0 * np.log(x) + rng.normal(scale=1.0, size=n)

X_raw = x.reshape(-1, 1)                                  # sklearn wants 2-D
X_log = np.log1p(x).reshape(-1, 1)                        # log1p = log(1 + x)

### Step 2 — Compare the raw and logged distributions

Side-by-side histograms show what the transform does to the feature's *shape*. The raw histogram is a tall spike near zero with a long thin tail to the right; the `log1p(x)` histogram is a roughly symmetric bell. This reshaping is the whole point of the log transform.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))

# Left: the raw feature -- a spike near 0 with a long right tail.
ax[0].hist(x, bins=40, color="#d9534f", edgecolor="white")
ax[0].set_title("RAW feature x (right-skewed, heavy tail)")
ax[0].set_xlabel("x")
ax[0].set_ylabel("count")

# Right: log1p(x) -- a much more symmetric, bell-like shape.
ax[1].hist(np.log1p(x), bins=40, color="#5cb85c", edgecolor="white")
ax[1].set_title("log1p(x)  (symmetric / ~Gaussian)")
ax[1].set_xlabel("log1p(x)")
ax[1].set_ylabel("count")

plt.tight_layout()
plt.show()

### Step 3 — Fit the same linear model on raw vs logged x

We fit `LinearRegression` twice: once on the raw `x`, once on `log1p(x)`. On the raw feature a straight line cannot follow a log-shaped curve and the handful of giant `x` values drag the fit around, so R^2 is poor. On the logged feature the relationship is linear, so R^2 is high. We also keep each model's residuals for the diagnostic plot.

In [ ]:
# Fit on RAW x: a straight line can't follow a log-shaped curve -> poor R^2.
raw_model = LinearRegression().fit(X_raw, y)
y_pred_raw = raw_model.predict(X_raw)
r2_raw = r2_score(y, y_pred_raw)
resid_raw = y - y_pred_raw
print(f"PROBLEM (raw x):        R^2 = {r2_raw:.3f}")

# Fit the SAME model on log1p(x): now the relation is linear -> high R^2.
log_model = LinearRegression().fit(X_log, y)
y_pred_log = log_model.predict(X_log)
r2_log = r2_score(y, y_pred_log)
resid_log = y - y_pred_log
print(f"FIX (log1p(x) feature): R^2 = {r2_log:.3f}")

### Step 4 — Read the residual-vs-fitted diagnostic

Residual-vs-fitted plots reveal whether the errors are well-behaved. On the raw fit (left) the residuals **funnel** — their spread grows with the fitted value (heteroscedastic), a sign the model is systematically wrong. On the logged fit (right) the residuals form an even, flat band around zero. The final line prints the before/after R^2.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4), sharey=True)

# Left: raw residuals FUNNEL -- spread grows with the fitted value.
ax[0].scatter(y_pred_raw, resid_raw, s=12, alpha=0.6, color="#d9534f")
ax[0].axhline(0, color="black", lw=1)
ax[0].set_title(f"RAW: residuals FUNNEL  (R^2={r2_raw:.2f})")
ax[0].set_xlabel("fitted value")
ax[0].set_ylabel("residual (y - prediction)")

# Right: logged residuals form an EVEN, flat band around 0.
ax[1].scatter(y_pred_log, resid_log, s=12, alpha=0.6, color="#5cb85c")
ax[1].axhline(0, color="black", lw=1)
ax[1].set_title(f"FIXED: residuals EVEN  (R^2={r2_log:.2f})")
ax[1].set_xlabel("fitted value")

plt.tight_layout()
plt.show()

# One-line before/after summary.
print(f"PROBLEM (raw): {r2_raw:.3f}   ->   FIX (log1p feature): {r2_log:.3f}")

## Reference implementation — pandas / numpy / scikit-learn / scipy

### Step 1 — Yelp review counts: does the log feature predict stars better?

We load the Yelp business data and add a `log_review_count = log10(review_count + 1)` column (the `+1` so a count of 0 maps to `log10(1) = 0`). Then we compare 10-fold cross-validated R^2 of a one-feature linear regression on the raw count versus the logged count. The book's result: both sit around -0.037 — the histogram is reshaped a lot, but this weak single-feature model barely changes.

In [ ]:
import pandas as pd
import numpy as np
import json
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score

# ---------------------------------------------------------------
# Example 1: Yelp business review counts (Yelp Dataset Challenge)
#   data: github.com/alicezheng/feature-engineering-book
# ---------------------------------------------------------------
biz_file = open('yelp_academic_dataset_business.json')
biz_df = pd.DataFrame([json.loads(x) for x in biz_file.readlines()])
biz_file.close()

# Log transform: add 1 first so review_count == 0 -> log10(1) == 0
biz_df['log_review_count'] = np.log10(biz_df['review_count'] + 1)

# Does the log feature predict star rating better than the raw count?
# 10-fold cross-validated R^2 of a one-feature linear regression.
m_orig = LinearRegression()
scores_orig = cross_val_score(m_orig, biz_df[['review_count']],
                              biz_df['stars'], cv=10)

m_log = LinearRegression()
scores_log = cross_val_score(m_log, biz_df[['log_review_count']],
                             biz_df['stars'], cv=10)

print("R^2 raw review_count : %0.5f (+/- %0.5f)"
      % (scores_orig.mean(), scores_orig.std() * 2))
print("R^2 log_review_count : %0.5f (+/- %0.5f)"
      % (scores_log.mean(), scores_log.std() * 2))
# Book result: both about -0.037 -- the histogram is reshaped a lot,
# but this weak single-feature model barely changes.

### Step 2 — Online News word counts: a small but real lift

The same recipe on the UCI Online News Popularity data, logging the `n_tokens_content` (article word-count) feature and predicting `shares`. Here the log helps a little more: cross-validated R^2 improves from -0.00242 raw to -0.00114 logged, because the log made the word-count feature much more Gaussian.

In [ ]:
# ---------------------------------------------------------------
# Example 2: UCI Online News Popularity word-count feature
#   archive.ics.uci.edu/ml/datasets/Online+News+Popularity
# ---------------------------------------------------------------
news_df = pd.read_csv('OnlineNewsPopularity.csv', delimiter=', ')

news_df['log_n_tokens_content'] = np.log10(news_df['n_tokens_content'] + 1)

m_orig = LinearRegression()
scores_orig = cross_val_score(m_orig, news_df[['n_tokens_content']],
                              news_df['shares'], cv=10)
m_log = LinearRegression()
scores_log = cross_val_score(m_log, news_df[['log_n_tokens_content']],
                             news_df['shares'], cv=10)

print("R^2 raw n_tokens_content : %0.5f" % scores_orig.mean())
print("R^2 log n_tokens_content : %0.5f" % scores_log.mean())
# Book result: improves from -0.00242 (raw) to -0.00114 (logged):
# the log made the word-count feature much more Gaussian and slightly
# improved the linear regression's R^2.

### Step 3 — Box-Cox picks the exponent for you

Box-Cox generalizes the log: it searches over a power `lambda` and picks the one that makes the feature most Gaussian (by maximum likelihood). It needs strictly positive data, so we feed `review_count + 1`. A returned `lambda` near 0 means the best transform is essentially a log — confirming the log was a sensible manual choice.

In [ ]:
# ---------------------------------------------------------------
# Example 3: Box-Cox on the Yelp review counts (needs positive data,
#   so feed review_count + 1). boxcox picks lambda by max likelihood.
# ---------------------------------------------------------------
rc_log = np.log10(biz_df['review_count'] + 1)
rc_bc, bc_params = stats.boxcox(biz_df['review_count'] + 1)
print("Box-Cox lambda = %0.4f" % bc_params)   # ~0 means ~ a log transform

## Visualize the data & results

_Take a heavy-tailed positive feature (the 'mean area' of breast-cancer tumors). Does a log transform turn its lopsided distribution into something more symmetric and bell-shaped?_

### Step 1 — Grab a heavy-tailed real feature and histogram it

We take tumor `mean area` (column 3) from the breast-cancer data — strictly positive and right-skewed — and subsample 60 points for a clean histogram. We bin the raw feature into 7 equal-width bins to capture its lopsided shape before transforming.

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer

# A heavy-tailed positive feature: tumor 'mean area' (column 3).
bc = load_breast_cancer()
area = bc.data[:, 3]                      # strictly positive, right-skewed

# Subsample to 60 points for a clean in-browser histogram.
rng = np.random.RandomState(0)
area = rng.choice(area, size=60, replace=False)

# BEFORE: histogram of the raw feature (7 equal-width bins).
raw_counts, raw_edges = np.histogram(area, bins=7)

### Step 2 — Log-transform and compare skew

We take `log10(area)` (all values are positive here, so no `+1` is needed) and re-bin. Comparing the skewness coefficient before and after shows the log pulling the right tail in: the raw feature has a large positive skew, the logged feature is much closer to symmetric. The bin counts make the same point — the log spreads the data out into a more Gaussian shape.

In [ ]:
# AFTER: histogram of log10(area). All values > 0, so no +1 needed here;
# for count features with zeros you would use np.log10(x + 1).
log_area = np.log10(area)
log_counts, log_edges = np.histogram(log_area, bins=7)

# Compare skewness before and after the log transform.
raw_skew = ((area - area.mean())**3).mean() / area.std()**3
log_skew = ((log_area - log_area.mean())**3).mean() / log_area.std()**3

print("raw skew (right tail):", round(float(raw_skew), 3))
print("log skew (more symmetric):", round(float(log_skew), 3))
print("raw  bin counts:", raw_counts.tolist())
print("log  bin counts:", log_counts.tolist())
# Raw distribution is right-skewed; log10 pulls the tail in and the
# histogram becomes far more symmetric / Gaussian -- exactly the book's point.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** A feature page_views is a count that ranges from 0 to about 4,000,000, with most pages under 100 views. You want to feed it to a linear regression. Your colleague writes np.log10(df['page_views']) and the pipeline crashes on some rows. What broke, what is the one-character-idea fix, and why does logging help this feature at all?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Find what crashed. — _Some pages have 0 views, and $\log_{10}(0)=-\infty$ (NumPy emits $-\infty$ / a warning, and downstream math on $-\infty$ blows up). Any zero-containing count feature hits this._
- Apply the book's fix. — _Add 1 before logging: np.log10(df['page_views'] + 1) (or np.log1p). Now a count of 0 maps to $\log_{10}(1)=0$, finite and sensible. The "+1" is the whole trick._
- Explain why logging is worth it here. — _Page views span six orders of magnitude with a giant tail. The log compresses the millions-of-views giants and spreads the under-100 crowd, turning a spike-plus-tail into a more symmetric, more Gaussian feature that a linear model handles better._

**Answer:** The crash is $\log_{10}(0)=-\infty$ — the count feature has zeros. The fix is the book's $\log_{10}(x+1)$: np.log10(df['page_views'] + 1), which sends 0 to 0 instead of $-\infty$. Logging helps because page_views is heavy-tailed over many orders of magnitude; the log squashes the giants and stretches the small bulk, producing a more symmetric, more Gaussian feature that linear regression prefers.

</details>

**Problem 2.** You run scipy.stats.boxcox on a strictly-positive feature and it returns $\lambda \approx 0.02$. (a) What transform is Box-Cox essentially applying? (b) What would $\lambda \approx 0.5$ have meant instead? (c) A teammate fits $\lambda$ on the full dataset (train + test together). Why is that a mistake?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Read $\lambda \approx 0$. — _Box-Cox at $\lambda=0$ is exactly $\ln(x)$. A fitted $\lambda\approx 0.02$ means maximum likelihood decided the feature is heavy-tailed enough that an essentially-log transform makes it most Gaussian._
- Contrast $\lambda \approx 0.5$. — _$\lambda=0.5$ is a (shifted, scaled) square root — a gentler squash, the natural variance-stabilizer for milder, Poisson-like count data._
- Spot the leak. — _$\lambda$ is a parameter estimated from data. Fitting it on test rows lets test information bleed into the transform, so your reported test score is optimistic. Fit $\lambda$ on train only, then apply that fixed $\lambda$ to validation/test — like any scaler._

**Answer:** (a) $\lambda\approx 0$ is the log: Box-Cox at $\lambda=0$ is $\ln(x)$, so it's applying essentially a natural-log transform. (b) $\lambda\approx 0.5$ would be a square-root-like squash — gentler, the variance-stabilizing choice for milder Poisson-like counts. (c) Fitting $\lambda$ on train+test leaks test information into a learned parameter, inflating the test score. Estimate $\lambda$ on the training set only and reuse that same $\lambda$ on validation and test.

</details>